# Exploratory Data Analysis

This notebook profiles the PhonePe district data before opportunity scoring. It covers data quality, univariate distributions, bivariate relationships, multivariate patterns, outliers, and the business implications for merchant expansion. Portfolio charts are saved to `reports/eda_charts/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from phonepe_analytics.metrics import add_growth_metrics, add_ratio_metrics

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
CHARTS = ROOT / "reports" / "eda_charts"
CHARTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

district = pd.read_csv(DATA / "district_quarter.csv")
state = pd.read_csv(DATA / "state_quarter.csv")
categories = pd.read_csv(DATA / "state_transaction_categories.csv")

district = add_growth_metrics(add_ratio_metrics(district), ["state", "district"])
state = add_growth_metrics(add_ratio_metrics(state), ["state"])

latest_period = int(district["period_id"].max())
latest = district[district["period_id"].eq(latest_period)].copy()
latest_state = state[state["period_id"].eq(latest_period)].copy()
latest_label = f"{int(latest['year'].max())} Q{int(latest['quarter'].max())}"

def insight(*lines: str) -> None:
    display(Markdown("**Insight**\n\n" + "\n".join(f"- {line}" for line in lines)))

def save(name: str) -> None:
    plt.tight_layout()
    plt.savefig(CHARTS / name, dpi=160, bbox_inches="tight")
    plt.show()

def profile(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        s = frame[column].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        rows.append({
            "variable": column,
            "count": len(s),
            "missing": frame[column].isna().sum(),
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(),
            "p25": q1,
            "p75": q3,
            "p90": s.quantile(0.90),
            "p95": s.quantile(0.95),
            "min": s.min(),
            "max": s.max(),
            "skewness": s.skew(),
            "iqr_outliers": ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum(),
        })
    return pd.DataFrame(rows).set_index("variable")


## 1. Coverage and completeness

The first check confirms the analytical grain and whether the latest quarter has the core fields needed for fair district comparisons.


In [ ]:
coverage = district.groupby(["year", "quarter"], as_index=False).agg(
    rows=("district", "size"),
    states=("state", "nunique"),
    districts=("district", "nunique"),
    merchant_missing=("registered_merchants", lambda s: s.isna().sum()),
)
missing = latest[
    ["transaction_count", "transaction_amount", "registered_users", "registered_merchants"]
].isna().sum()

display(coverage.tail(8))
display(missing.rename("missing_values").to_frame())
insight(
    f"The latest period is {latest_label} with {len(latest):,} district rows across {latest['state'].nunique():,} states/UTs.",
    f"Core latest-quarter missing values total {int(missing.sum()):,}; incomplete districts should not enter opportunity ranking.",
    "Historical merchant nulls are source limitations and must not be converted to zero because that would create false penetration and growth signals.",
)


## 2. Univariate analysis

Univariate analysis shows the typical district, the degree of skew, the spread of growth rates, and where outliers make simple averages misleading.


In [ ]:
core = [
    "transaction_count", "transaction_amount", "registered_users", "registered_merchants",
    "average_transaction_value", "transactions_per_registered_user",
    "tpv_per_registered_user", "merchants_per_100k_users", "users_per_merchant",
]
growth = [
    "transaction_count_qoq", "transaction_amount_qoq", "registered_users_qoq",
    "registered_merchants_qoq", "transaction_yoy",
]

core_profile = profile(latest, core)
growth_profile = profile(latest, growth)
display(core_profile)
display(growth_profile)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, column in zip(
    axes.flat,
    ["transaction_count", "transaction_amount", "registered_users",
     "registered_merchants", "merchants_per_100k_users", "transaction_yoy"],
):
    values = latest[column].replace([np.inf, -np.inf], np.nan).dropna()
    if column in {"transaction_count", "transaction_amount", "registered_users", "registered_merchants"}:
        values = np.log1p(values)
        xlabel = f"log(1 + {column})"
    else:
        values = values.clip(values.quantile(0.01), values.quantile(0.99))
        xlabel = column
    sns.histplot(values, bins=35, kde=True, ax=ax)
    ax.set_xlabel(xlabel)
    ax.set_title(column.replace("_", " " ).title())

save("univariate_distributions.png")

skewed = core_profile["skewness"].abs().nlargest(3).index.tolist()
positive_yoy = latest["transaction_yoy"].gt(0).mean()
insight(
    f"The most skewed scale measures are {', '.join(skewed)}; medians and percentiles are more representative than means for these variables.",
    f"{positive_yoy:.1%} of districts have positive same-quarter transaction growth in {latest_label}.",
    "Scale and growth distributions justify percentile-based comparisons because a small number of very large or unusually fast-growing districts can dominate raw metrics.",
)


In [ ]:
latest_categories = categories[categories["period_id"].eq(categories["period_id"].max())]
mix = latest_categories.groupby("category", as_index=False)["transaction_count"].sum()
mix["share"] = mix["transaction_count"] / mix["transaction_count"].sum()
mix = mix.sort_values("share", ascending=False)
display(mix)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=mix, x="share", y="category")
plt.xlabel("Share of state-level category transactions")
plt.ylabel("")
plt.title(f"Transaction category mix | {latest_label}")
save("transaction_category_mix.png")

leader = mix.iloc[0]
insight(
    f"{leader['category']} is the largest state-level category at {leader['share']:.1%} of category transactions.",
    "Category mix matters because total PhonePe transaction growth is not automatically merchant-payment growth.",
    "District category data is unavailable, so the project should describe district totals as ecosystem activity rather than merchant-only transactions.",
)


## 3. Bivariate analysis

The next section tests the relationships behind the business story: market scale versus demand, merchant presence versus activity, and growth versus merchant penetration.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

pair = latest.dropna(subset=["registered_users", "transaction_count"])
sns.regplot(
    x=np.log1p(pair["registered_users"]),
    y=np.log1p(pair["transaction_count"]),
    scatter_kws={"alpha": 0.35, "s": 24},
    ax=axes[0, 0],
)
axes[0, 0].set_title("Users vs transactions")
axes[0, 0].set_xlabel("log(1 + registered users)")
axes[0, 0].set_ylabel("log(1 + transactions)")

pair = latest.dropna(subset=["registered_merchants", "transaction_count"])
sns.regplot(
    x=np.log1p(pair["registered_merchants"]),
    y=np.log1p(pair["transaction_count"]),
    scatter_kws={"alpha": 0.35, "s": 24},
    ax=axes[0, 1],
)
axes[0, 1].set_title("Merchants vs transactions")
axes[0, 1].set_xlabel("log(1 + registered merchants)")
axes[0, 1].set_ylabel("log(1 + transactions)")

sns.scatterplot(
    data=latest,
    x="merchants_per_100k_users",
    y="transaction_yoy",
    size="registered_users",
    sizes=(15, 160),
    alpha=0.4,
    legend=False,
    ax=axes[1, 0],
)
axes[1, 0].axhline(0, linestyle="--", linewidth=1)
axes[1, 0].set_title("Growth vs merchant penetration")

sns.scatterplot(
    data=latest,
    x="registered_users_qoq",
    y="registered_merchants_qoq",
    alpha=0.4,
    ax=axes[1, 1],
)
axes[1, 1].axline((0, 0), slope=1, linestyle="--", linewidth=1)
axes[1, 1].set_title("User growth vs merchant growth")

save("bivariate_relationships.png")

rho_users = latest[["registered_users", "transaction_count"]].corr(method="spearman").iloc[0, 1]
rho_merchants = latest[["registered_merchants", "transaction_count"]].corr(method="spearman").iloc[0, 1]
rho_penetration = latest[["merchants_per_100k_users", "transaction_yoy"]].corr(method="spearman").iloc[0, 1]
growth_gap_share = (latest["registered_users_qoq"] > latest["registered_merchants_qoq"]).mean()

insight(
    f"Registered users and transactions have a Spearman association of {rho_users:.3f}; market size explains a meaningful part of demand.",
    f"Registered merchants and transactions have a Spearman association of {rho_merchants:.3f}; this is association, not evidence that merchant registrations cause transactions.",
    f"Merchant penetration and transaction YoY growth have a Spearman association of {rho_penetration:.3f}.",
    f"User growth exceeds merchant growth in {growth_gap_share:.1%} of latest-quarter districts, highlighting areas worth deeper comparison.",
)


## 4. Multivariate analysis

Multivariate analysis checks whether score inputs overlap and combines demand growth, merchant penetration, and market scale in one view.


In [ ]:
corr_columns = [
    "transaction_count", "transaction_amount", "registered_users", "registered_merchants",
    "average_transaction_value", "transactions_per_registered_user",
    "merchants_per_100k_users", "transaction_yoy", "transaction_count_qoq",
    "registered_users_qoq", "registered_merchants_qoq",
]
corr = latest[corr_columns].corr(method="spearman")

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f")
plt.title(f"Spearman correlation matrix | {latest_label}")
save("correlation_heatmap.png")

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pair = upper.abs().stack().idxmax()
strong_value = corr.loc[strong_pair[0], strong_pair[1]]
insight(
    f"The strongest selected pair is {strong_pair[0]} vs {strong_pair[1]} with Spearman {strong_value:.3f}.",
    "Highly correlated inputs should not receive independent heavy weights in an opportunity score because that double-counts the same underlying characteristic.",
    "The correlation matrix is therefore both EDA and a control on score design.",
)


In [ ]:
multi = latest.dropna(
    subset=["transaction_yoy", "merchants_per_100k_users", "registered_users", "transaction_count"]
).copy()
growth_cut = multi["transaction_yoy"].median()
penetration_cut = multi["merchants_per_100k_users"].median()

multi["eda_segment"] = np.select(
    [
        multi["transaction_yoy"].ge(growth_cut) & multi["merchants_per_100k_users"].lt(penetration_cut),
        multi["transaction_yoy"].ge(growth_cut) & multi["merchants_per_100k_users"].ge(penetration_cut),
        multi["transaction_yoy"].lt(growth_cut) & multi["merchants_per_100k_users"].lt(penetration_cut),
    ],
    ["High growth / lower penetration", "High growth / higher penetration", "Lower growth / lower penetration"],
    default="Lower growth / higher penetration",
)

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=multi,
    x="merchants_per_100k_users",
    y="transaction_yoy",
    hue="eda_segment",
    size="registered_users",
    sizes=(18, 240),
    alpha=0.55,
)
plt.axvline(penetration_cut, linestyle="--", linewidth=1)
plt.axhline(growth_cut, linestyle="--", linewidth=1)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title(f"Growth, merchant penetration, and user scale | {latest_label}")
save("multivariate_opportunity_quadrants.png")

summary = multi.groupby("eda_segment", as_index=False).agg(
    districts=("district", "size"),
    median_users=("registered_users", "median"),
    median_transactions=("transaction_count", "median"),
    median_yoy_growth=("transaction_yoy", "median"),
    median_merchants_per_100k=("merchants_per_100k_users", "median"),
)
display(summary)

candidate_count = int(summary.loc[
    summary["eda_segment"].eq("High growth / lower penetration"), "districts"
].iloc[0])
insight(
    f"{candidate_count:,} districts fall into the exploratory high-growth/lower-penetration quadrant.",
    "These markets are the most relevant EDA candidates because demand momentum and relative merchant-registration gaps appear together.",
    "Bubble size adds commercial scale, preventing a tiny high-growth district from being treated as equivalent to a large market.",
    "The quadrants are exploratory; the final recommendation still requires eligibility thresholds and sensitivity analysis.",
)


In [ ]:
national = state.groupby(["year", "quarter", "period_id"], as_index=False).agg(
    transaction_count=("transaction_count", "sum"),
    registered_users=("registered_users", "sum"),
    registered_merchants=("registered_merchants", lambda s: s.sum(min_count=1)),
).sort_values("period_id")
national["period_label"] = national["year"].astype(str) + " Q" + national["quarter"].astype(str)

for column in ["transaction_count", "registered_users", "registered_merchants"]:
    first = national[column].first_valid_index()
    national[f"{column}_index"] = national[column] / national.loc[first, column] * 100

indexed = national.melt(
    id_vars=["period_id", "period_label"],
    value_vars=["transaction_count_index", "registered_users_index", "registered_merchants_index"],
    var_name="metric",
    value_name="index",
)
indexed["metric"] = indexed["metric"].str.replace("_index", "").str.replace("_", " " ).str.title()

plt.figure(figsize=(11, 5.5))
sns.lineplot(data=indexed, x="period_id", y="index", hue="metric", linewidth=2)
ticks = national.iloc[::4]
plt.xticks(ticks["period_id"], ticks["period_label"], rotation=45, ha="right")
plt.ylabel("Index (first available quarter = 100)")
plt.xlabel("")
plt.title("Relative growth of transactions, users, and merchants")
save("multivariate_national_growth_index.png")

last = national.iloc[-1]
insight(
    f"By {latest_label}, the transaction index is {last['transaction_count_index']:,.1f}, user index {last['registered_users_index']:,.1f}, and merchant index {last['registered_merchants_index']:,.1f}.",
    "Indexed growth separates pace from size and shows whether usage is deepening faster than registrations.",
    "Merchant history must be interpreted from its first available period because earlier merchant observations are incomplete in the source.",
)


## 5. Outliers and analytical implications

Outliers in geographic payment data are often real large markets rather than bad records. IQR checks are used to identify them, not automatically delete them.


In [ ]:
outlier_columns = [
    "transaction_count", "registered_users", "registered_merchants",
    "merchants_per_100k_users", "transaction_yoy", "transactions_per_registered_user",
]
records = []
for column in outlier_columns:
    s = latest[column].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    flagged = latest.loc[
        latest[column].lt(q1 - 1.5 * iqr) | latest[column].gt(q3 + 1.5 * iqr),
        ["state", "district", column],
    ].copy()
    flagged["metric"] = column
    records.append(flagged.rename(columns={column: "value"}))

outliers = pd.concat(records, ignore_index=True)
display(outliers.groupby("metric").size().rename("outlier_districts").to_frame())
display(outliers.sort_values(["metric", "value"], ascending=[True, False]).head(25))

insight(
    f"IQR screening flags {len(outliers):,} metric-level observations; these are review candidates rather than automatic deletions.",
    "Extreme growth rates deserve special caution because small prior-period denominators can magnify percentage changes.",
    "Percentile normalization and minimum scale filters reduce outlier dominance while preserving commercially important large markets.",
)


## 6. EDA conclusions

- **Scale is uneven.** Medians, percentiles, and log-scale views are more representative than means for district comparisons.
- **Demand and market size move together.** A pure transaction ranking would mostly identify the largest markets.
- **Low merchant penetration is not enough.** It becomes useful only when combined with healthy demand, user scale, and growth.
- **YoY and QoQ serve different purposes.** YoY is better for structural momentum; QoQ captures recent acceleration or slowdown.
- **Composite inputs can overlap.** Correlation checks should guide score weights to avoid double-counting.
- **Outliers should be controlled rather than removed blindly.**
- **The evidence is observational.** The analysis identifies markets worth investigating; it does not prove that merchant acquisition will cause transaction growth.

The next step is the opportunity-scoring and sensitivity analysis on the comparable district universe.
